In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
from datetime import date as dt
import plotly.express as px
import json
import warnings
warnings.filterwarnings('ignore')

In [2]:
url = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"

nifty500_list = pd.read_csv(url)

print(nifty500_list.head())
print(len(nifty500_list))

               Company Name                Industry     Symbol Series  \
0          360 ONE WAM Ltd.      Financial Services     360ONE     EQ   
1             3M India Ltd.             Diversified    3MINDIA     EQ   
2            ABB India Ltd.           Capital Goods        ABB     EQ   
3                  ACC Ltd.  Construction Materials        ACC     EQ   
4  ACME Solar Holdings Ltd.                   Power  ACMESOLAR     EQ   

      ISIN Code  
0  INE466L01038  
1  INE470A01017  
2  INE117A01022  
3  INE012A01025  
4  INE622W01025  
500


In [3]:
to_remove = ['DUMMYVEDL1', 'DUMMYVEDL3', 'DUMMYVEDL4', 'DUMMYVEDL2']
nifty500_list = nifty500_list[~nifty500_list["Symbol"].isin(to_remove)]

print(nifty500_list['Symbol'].is_unique)
nifty500_list['YahooTicker'] = nifty500_list['Symbol'] + '.NS'

True


In [4]:
nifty500_list

,Company Name,Industry,Symbol,Series,ISIN Code,YahooTicker
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038,360ONE.NS
1,3M India Ltd.,Diversified,3MINDIA,EQ,INE470A01017,3MINDIA.NS
2,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022,ABB.NS
3,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025,ACC.NS
4,ACME Solar Holdings Ltd.,Power,ACMESOLAR,EQ,INE622W01025,ACMESOLAR.NS
...,...,...,...,...,...,...
495,Zen Technologies Ltd.,Capital Goods,ZENTEC,EQ,INE251B01027,ZENTEC.NS
496,Zensar Technolgies Ltd.,Information Technology,ZENSARTECH,EQ,INE520A01027,ZENSARTECH.NS
497,Zydus Lifesciences Ltd.,Healthcare,ZYDUSLIFE,EQ,INE010B01027,ZYDUSLIFE.NS
498,Zydus Wellness Ltd.,Fast Moving Consumer Goods,ZYDUSWELL,EQ,INE768C01028,ZYDUSWELL.NS


In [49]:
user_stocklist = ['BANKINDIA.NS', 'BHARTIARTL.NS', 'GAIL.NS', 'GODREJIND.NS'] # We'll ask for it in deployment
user_period = 3 # We'll ask for it too

In [50]:
end = dt.today()
start = dt(dt.today().year-user_period, dt.today().month, dt.today().day)

In [51]:
include_market = False # This will be a radio button in web app

In [52]:
nifty500 = yf.download(tickers= user_stocklist + ["^CRSLDX"], # This will include market(Nifty500) data too
                start= start,
                end= end)

[*********************100%***********************]  5 of 5 completed


In [53]:
nifty500.sort_index(ascending = False).stack()

Price                            Close          High           Low  \
Date       Ticker                                                    
2026-07-10 BANKINDIA.NS     144.899994    146.899994    139.500000   
           BHARTIARTL.NS   1920.400024   1930.000000   1906.000000   
           GAIL.NS          173.729996    174.470001    171.000000   
           GODREJIND.NS    1417.699951   1443.000000   1210.400024   
           ^CRSLDX        23348.400391  23358.349609  23234.449219   
...                                ...           ...           ...   
2023-07-11 BANKINDIA.NS      70.945618     72.087691     70.717203   
           BHARTIARTL.NS    875.581848    876.808978    870.084270   
           GAIL.NS           98.473152     98.743431     96.986594   
           GODREJIND.NS     496.250000    503.000000    491.000000   
           ^CRSLDX        16648.000000  16698.800781  16597.550781   

Price                             Open      Volume  
Date       Ticker                                   
2026-07-10 BANKINDIA.NS     139.800003  14184717.0  
           BHARTIARTL.NS   1930.000000   6420887.0  
           GAIL.NS          171.000000   3236963.0  
           GODREJIND.NS    1219.500000   2661745.0  
           ^CRSLDX        23239.699219  23139200.0  
...                                ...         ...  
2023-07-11 BANKINDIA.NS      71.813591   6322004.0  
           BHARTIARTL.NS    873.716587   3353977.0  
           GAIL.NS           98.157816  10380385.0  
           GODREJIND.NS     491.000000    164183.0  
           ^CRSLDX        16610.949219  19751600.0  

[3715 rows x 5 columns]

In [54]:
df = nifty500['Close']

In [55]:
df.rename(columns = {'^CRSLDX':'NIFTY500'}, inplace = True)

In [56]:
df.head()

Ticker,BANKINDIA.NS,BHARTIARTL.NS,GAIL.NS,GODREJIND.NS,NIFTY500
Date,,,,,
2023-07-11,70.945618,875.581848,98.473152,496.250000,16648.000000
2023-07-12,72.453163,873.667480,99.644371,497.250000,16641.050781
2023-07-13,70.580154,868.857117,97.392006,489.649994,16623.900391
2023-07-14,71.128342,870.280579,99.103798,489.200012,16765.449219
2023-07-17,72.864304,862.034241,97.842476,489.450012,16872.050781


In [57]:
df.tail()

Ticker,BANKINDIA.NS,BHARTIARTL.NS,GAIL.NS,GODREJIND.NS,NIFTY500
Date,,,,,
2026-07-06,139.800003,1925.699951,175.029999,1209.099976,23433.949219
2026-07-07,142.410004,1925.800049,174.009995,1212.800049,23368.849609
2026-07-08,135.119995,1888.099976,169.240005,1216.400024,22908.050781
2026-07-09,138.529999,1931.099976,170.199997,1219.199951,23081.150391
2026-07-10,144.899994,1920.400024,173.729996,1417.699951,23348.400391


In [58]:
def interactive_plot(df):
    fig = px.line()

    for col in df.columns:
        if col == 'NIFTY500' and include_market == False:
            continue
        fig.add_scatter(x= df.index, y= df[col], name = col)
    fig.update_layout(width= 450, margin= dict(l=20, r=20, t=50, b=20), legend= dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
    return fig

In [59]:
interactive_plot(df)

In [60]:
def normalize_values(df):
    df_changed = df.copy()
    for col in df_changed.columns:
        df_changed[col] = df_changed[col]/df_changed[col].iloc[0]
    return df_changed

In [61]:
normalize_values(df)

Ticker,BANKINDIA.NS,BHARTIARTL.NS,GAIL.NS,GODREJIND.NS,NIFTY500
Date,,,,,
2023-07-11,1.000000,1.000000,1.000000,1.000000,1.000000
2023-07-12,1.021249,0.997814,1.011894,1.002015,0.999583
2023-07-13,0.994849,0.992320,0.989021,0.986700,0.998552
2023-07-14,1.002576,0.993945,1.006404,0.985793,1.007055
2023-07-17,1.027044,0.984527,0.993595,0.986297,1.013458
...,...,...,...,...,...
2026-07-06,1.970523,2.199337,1.777439,2.436474,1.407613
2026-07-07,2.007312,2.199452,1.767081,2.443930,1.403703
2026-07-08,1.904557,2.156395,1.718641,2.451184,1.376024


In [62]:
interactive_plot(normalize_values(df))

In [63]:
def daily_return(df):
    return df.pct_change() * 100

In [64]:
print(daily_return(df))

Ticker      BANKINDIA.NS  BHARTIARTL.NS   GAIL.NS  GODREJIND.NS  NIFTY500
Date                                                                     
2023-07-11           NaN            NaN       NaN           NaN       NaN
2023-07-12      2.124931      -0.218639  1.189379      0.201511 -0.041742
2023-07-13     -2.585130      -0.550594 -2.260404     -1.528407 -0.103061
2023-07-14      0.776688       0.163832  1.757631     -0.091899  0.851478
2023-07-17      2.440605      -0.947549 -1.272728      0.051104  0.635841
...                  ...            ...       ...           ...       ...
2026-07-06     -2.950364       0.800876  0.505314     -0.057867  0.569924
2026-07-07      1.866953       0.005198 -0.582760      0.306019 -0.277800
2026-07-08     -5.119028      -1.957632 -2.741216      0.296832 -1.971851
2026-07-09      2.523685       2.277422  0.567237      0.230181  0.755628
2026-07-10      4.598278      -0.554086  2.074030     16.281169  1.157871

[743 rows x 5 columns]


In [76]:
def cal_beta(stock_daily_return, stock):
    temp = stock_daily_return[['NIFTY500', stock]].dropna()
    
    ''' Stock_return = b * Market_return + a'''
    b, a = np.polyfit(temp['NIFTY500'], temp[stock], 1)

    return b, a

In [82]:
beta = {}
alpha = {}
df_daily_return = daily_return(df)
for col in df_daily_return.columns:
    if col != 'NIFTY500':
        b, a = cal_beta(df_daily_return, col)
        beta[col] = b
        alpha[col] = a

betaframe = pd.DataFrame({'Stock': list(beta.keys()),
                          'Beta Values' : list(beta.values())})

betaframe

,Stock,Beta Values
0,BANKINDIA.NS,1.491057
1,BHARTIARTL.NS,0.687092
2,GAIL.NS,1.536357
3,GODREJIND.NS,0.991708


In [86]:
# Risk Free return : The Indian 10-year government bond yield stands at approximately 6.72%
rf = 6.72
rm = df_daily_return['NIFTY500'].mean() * 252 #About 252 trading days in a year

return_val = []

for stock, value in beta.items():
    return_val.append(round(rf + value * (rm - rf), 2))

return_df = pd.DataFrame({'Stock' : beta.keys(),
                          '% Return' : return_val})

return_df

,Stock,% Return
0,BANKINDIA.NS,15.08
1,BHARTIARTL.NS,10.57
2,GAIL.NS,15.33
3,GODREJIND.NS,12.28


In [2]:
def get_list_nifty500():
    url = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"
    nifty500_list = pd.read_csv(url)

    to_remove = ['DUMMYVEDL1', 'DUMMYVEDL3', 'DUMMYVEDL4', 'DUMMYVEDL2']
    nifty500_list = nifty500_list[~nifty500_list["Symbol"].isin(to_remove)]
    nifty500_list['YahooTicker'] = nifty500_list['Symbol'] + '.NS'

    return nifty500_list

nifty500_list = get_list_nifty500()

company = nifty500_list['Company Name'].to_list()
symbol = nifty500_list['Symbol'].to_list()
yticker = nifty500_list['YahooTicker'].to_list()

niftydict = {}
for com, sym, ytick in zip(company, symbol, yticker):
    niftydict[com] = [sym, ytick]

This thing I ran once to create the json files. I used it in case the yf.Ticker().info reached its limit. The data might be state but the app won't crash

In [3]:
stock_info = {}
count = 0
for company, (_, ticker) in niftydict.items():
    count += 1
    # print(f"Fetching {ticker}...")

    try:
        info = yf.Ticker(ticker).get_info()

        fundamentals = {
            "Market Cap": (
                f"₹{info["marketCap"] / 1e7:,.0f} Cr"
                if info.get("marketCap") is not None
                else None
            ),
            "P/E (TTM)": (round(info.get("trailingPE"), 2)
                    if info.get("trailingPE") is not None
                    else None
            ),
            "P/B": (round(info.get("priceToBook"), 2)
                if info.get("priceToBook") is not None
                    else None
            ),
            "ROE (%)": (
                round(info["returnOnEquity"] * 100, 2)
                if info.get("returnOnEquity") is not None
                else None
            ),
            "EPS (TTM)": info.get("trailingEps"),
            "Dividend Yield (%)": (
                round(info["dividendYield"] * 100, 2)
                if info.get("dividendYield") is not None
                else None
            ),
            "Debt-to-Equity": info.get("debtToEquity"),
            "52 Week High": info.get("fiftyTwoWeekHigh"),
            "52 Week Low": info.get("fiftyTwoWeekLow"),
            "Beta": info.get("beta"),
            "Sector": info.get("sector"),
            "Industry": info.get("industry"),
            "Website": info.get("website"),
            "Long Business Summary": info.get("longBusinessSummary"),
            "Full Time Employees": info.get("fullTimeEmployees"),
            "longBusinessSummary": info.get('longBusinessSummary'),
            "fullTimeEmployees": info.get('fullTimeEmployees'),
            'website': info.get('website')
        }

        stock_info[ticker] = fundamentals

    except Exception as e:
        print(f"Failed: {ticker} -> {e}")
        stock_info[ticker] = None


with open("stock_fundamentals.json", "w", encoding="utf-8") as f:
    json.dump(stock_info, f, indent=4, ensure_ascii=False)

print("JSON file created successfully.", count)

JSON file created successfully. 500
